# Results Analysis — PatchCore on MVTec AD

This notebook loads the evaluation outputs produced by `scripts/evaluate.py` and
`scripts/benchmark.py` and answers three questions:

1. **How close does this implementation get to the paper numbers?**
2. **Which categories are hardest / easiest?**
3. **How sensitive are results to coreset_ratio and num_neighbors?** (ablation)

> **Pre-requisite:** run `make evaluate-all` (or `python scripts/evaluate.py --category all`)
> before executing this notebook.

---

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

OUTPUTS = ROOT / "outputs"
print(f"Outputs dir: {OUTPUTS}")

## 1. Load Results

In [ ]:
results_path = OUTPUTS / "all_results.json"

if not results_path.exists():
    print("No results found. Run 'make evaluate-all' first.")
    results = []
else:
    with open(results_path) as f:
        results = json.load(f)
    print(f"Loaded results for {len(results)} categories")

In [ ]:
# Reference values from the PatchCore paper (Table 1, CVPR 2022)
PAPER = {
    "bottle":      {"image_auroc": 1.000, "pixel_auroc": 0.981},
    "cable":       {"image_auroc": 0.995, "pixel_auroc": 0.985},
    "capsule":     {"image_auroc": 0.981, "pixel_auroc": 0.988},
    "carpet":      {"image_auroc": 0.987, "pixel_auroc": 0.990},
    "grid":        {"image_auroc": 0.982, "pixel_auroc": 0.985},
    "hazelnut":    {"image_auroc": 1.000, "pixel_auroc": 0.987},
    "leather":     {"image_auroc": 1.000, "pixel_auroc": 0.992},
    "metal_nut":   {"image_auroc": 1.000, "pixel_auroc": 0.985},
    "pill":        {"image_auroc": 0.966, "pixel_auroc": 0.978},
    "screw":       {"image_auroc": 0.989, "pixel_auroc": 0.990},
    "tile":        {"image_auroc": 0.987, "pixel_auroc": 0.960},
    "toothbrush":  {"image_auroc": 1.000, "pixel_auroc": 0.987},
    "transistor":  {"image_auroc": 1.000, "pixel_auroc": 0.975},
    "wood":        {"image_auroc": 0.992, "pixel_auroc": 0.955},
    "zipper":      {"image_auroc": 0.994, "pixel_auroc": 0.987},
}

print(f"Paper mean Image AUROC : {np.mean([v['image_auroc'] for v in PAPER.values()]):.4f}")
print(f"Paper mean Pixel AUROC : {np.mean([v['pixel_auroc'] for v in PAPER.values()]):.4f}")

## 2. Implementation vs Paper

In [ ]:
if results:
    cats         = [r["category"]    for r in results]
    img_auroc    = [r["image_auroc"] for r in results]
    pxl_auroc    = [r.get("pixel_auroc") or 0 for r in results]
    paper_img    = [PAPER.get(c, {}).get("image_auroc", 0) for c in cats]
    paper_pxl    = [PAPER.get(c, {}).get("pixel_auroc", 0) for c in cats]

    x  = np.arange(len(cats))
    w  = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax, ours, paper, title in [
        (axes[0], img_auroc, paper_img, "Image-level AUROC"),
        (axes[1], pxl_auroc, paper_pxl, "Pixel-level AUROC"),
    ]:
        ax.bar(x - w/2, paper, width=w, label="Paper",  color="steelblue",    alpha=0.8)
        ax.bar(x + w/2, ours,  width=w, label="Ours",   color="darkorange",   alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(cats, rotation=40, ha="right", fontsize=8)
        ax.set_ylim(0.85, 1.01)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
        ax.set_title(title)
        ax.legend()
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Implementation vs Paper (PatchCore, WideResNet-50-2)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"Our mean Image AUROC : {np.mean(img_auroc):.4f}  (paper: {np.mean(paper_img):.4f})")
    print(f"Our mean Pixel AUROC : {np.mean(pxl_auroc):.4f}  (paper: {np.mean(paper_pxl):.4f})")

## 3. Category Difficulty Ranking

In [ ]:
if results:
    sorted_results = sorted(results, key=lambda r: r["image_auroc"])
    cats_s   = [r["category"]    for r in sorted_results]
    scores_s = [r["image_auroc"] for r in sorted_results]

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["tomato" if s < 0.98 else "mediumseagreen" for s in scores_s]
    bars   = ax.barh(cats_s, scores_s, color=colors)

    ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
    ax.set_xlim(0.85, 1.02)
    ax.axvline(x=0.98, color="gray", linestyle="--", linewidth=1, label="0.98 threshold")
    ax.set_xlabel("Image AUROC")
    ax.set_title("Category Difficulty (Image AUROC, sorted)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    easiest = sorted_results[-1]["category"]
    hardest = sorted_results[0]["category"]
    print(f"Easiest: {easiest}  ({sorted_results[-1]['image_auroc']:.4f})")
    print(f"Hardest: {hardest}  ({sorted_results[0]['image_auroc']:.4f})")

## 4. Score Distribution — Normal vs Anomalous

A well-calibrated model should have a clear separation between the score distributions of
normal and anomalous samples. Let's inspect this for one category.

In [ ]:
import torch
from src.data.dataset import MVTecADDataset
from src.models.patchcore import PatchCore
from src.utils.config import load_config

CATEGORY = "bottle"   # change to any trained category
config   = load_config(str(ROOT / "configs" / "default.yaml"))
model_path = OUTPUTS / CATEGORY / "model.pt"

if not model_path.exists():
    print(f"No trained model for '{CATEGORY}'. Run 'make train CATEGORY={CATEGORY}' first.")
else:
    model = PatchCore(
        backbone=config["model"]["backbone"],
        layers=config["model"]["layers"],
        pretrained=config["model"]["pretrained"],
        device="cpu",
    )
    model.load(str(model_path))

    ds = MVTecADDataset(
        root=config["dataset"]["root"],
        category=CATEGORY,
        split="test",
        image_size=config["dataset"]["image_size"],
        center_crop=config["dataset"]["center_crop"],
    )
    loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)

    normal_scores, anom_scores = [], []
    for batch in loader:
        scores, _ = model.predict(batch["image"])
        for s, lbl in zip(scores.numpy(), batch["label"].numpy()):
            (anom_scores if lbl == 1 else normal_scores).append(float(s))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(normal_scores, bins=30, alpha=0.6, color="steelblue",  label="Normal")
    ax.hist(anom_scores,   bins=30, alpha=0.6, color="tomato",     label="Anomalous")

    metrics_path = OUTPUTS / CATEGORY / "eval_metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            m = json.load(f)
        thr = m.get("image_threshold")
        if thr:
            ax.axvline(thr, color="black", linestyle="--", label=f"Threshold={thr:.3f}")

    ax.set_xlabel("Anomaly score")
    ax.set_ylabel("Count")
    ax.set_title(f"Score Distribution — {CATEGORY}")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Normal  scores:  mean={np.mean(normal_scores):.4f}  std={np.std(normal_scores):.4f}")
    print(f"Anomalous scores: mean={np.mean(anom_scores):.4f}  std={np.std(anom_scores):.4f}")

## 5. Ablation Study Results

Load results from `scripts/benchmark.py` (run `make benchmark` first).

In [ ]:
BENCHMARK_CAT = "bottle"
bench_path    = OUTPUTS / "benchmark" / f"{BENCHMARK_CAT}_ablation.json"

if not bench_path.exists():
    print(f"No benchmark results. Run 'make benchmark CATEGORY={BENCHMARK_CAT}' first.")
    bench = []
else:
    with open(bench_path) as f:
        bench = json.load(f)
    print(f"Loaded {len(bench)} benchmark runs")

In [ ]:
if bench:
    coreset_runs = [r for r in bench if r["experiment"] == "coreset_ratio"]
    knn_runs     = [r for r in bench if r["experiment"] == "num_neighbors"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Coreset ratio ─────────────────────────────────────────────────────────
    if coreset_runs:
        ax   = axes[0]
        x    = [r["coreset_ratio"]  for r in coreset_runs]
        img  = [r["image_auroc"]    for r in coreset_runs]
        pxl  = [r.get("pixel_auroc") or 0 for r in coreset_runs]
        mem  = [r["memory_bank_size"] for r in coreset_runs]

        ax.plot(x, img, "o-", color="steelblue",  label="Image AUROC")
        ax.plot(x, pxl, "s-", color="tomato",     label="Pixel AUROC")
        ax.set_xlabel("coreset_ratio")
        ax.set_ylabel("AUROC")
        ax.set_title("Effect of coreset_ratio")
        ax.legend()
        ax.grid(alpha=0.3)

        ax2 = ax.twinx()
        ax2.plot(x, [m / 1000 for m in mem], "^--", color="purple", alpha=0.5, label="Memory bank (k)")
        ax2.set_ylabel("Memory bank size (thousands)", color="purple")
        ax2.legend(loc="lower right")

    # ── num_neighbors ─────────────────────────────────────────────────────────
    if knn_runs:
        ax   = axes[1]
        x    = [r["num_neighbors"]  for r in knn_runs]
        img  = [r["image_auroc"]    for r in knn_runs]
        pxl  = [r.get("pixel_auroc") or 0 for r in knn_runs]

        ax.plot(x, img, "o-", color="steelblue",  label="Image AUROC")
        ax.plot(x, pxl, "s-", color="tomato",     label="Pixel AUROC")
        ax.set_xlabel("num_neighbors (k)")
        ax.set_ylabel("AUROC")
        ax.set_title("Effect of num_neighbors (k-NN)")
        ax.legend()
        ax.grid(alpha=0.3)

    fig.suptitle(f"Ablation Study — {BENCHMARK_CAT}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Key Observations

*(Fill in after running experiments — these are starter prompts)*

**Implementation vs paper:**
- The implementation reproduces the paper numbers closely on most categories.
- Any gap is likely due to randomness in coreset subsampling (seed differences) or
  minor differences in the ImageNet normalisation constants used.

**Hardest categories:**
- Categories with small, subtle defects (e.g. `screw`, `capsule`) tend to score lower
  on image AUROC because the anomaly signal is spread across fewer patches.
- Texture categories (carpet, grid) tend to score higher because defects change a
  large fraction of the image and are thus easier to distinguish from normal patches.

**Coreset ratio sensitivity:**
- AUROC degrades noticeably only at very small ratios (< 0.05). At 0.1 (default) the
  memory bank is already a well-representative sample of the training distribution.
- The memory bank grows linearly with the ratio — a 10x reduction in `coreset_ratio`
  gives ~10x smaller memory and ~10x faster inference with minimal accuracy loss.

**k-NN sensitivity:**
- Results are robust across a wide range of `k` values (3–15).
- Very small k (k=1) is more sensitive to noise; very large k over-smooths anomaly maps.
- The default k=9 sits comfortably in the stable region.